# 📖 Module 08: Prompting for RAG

## GenAI L2 Exam Preparation

**Topics Covered:**
- Prompting techniques: Zero-shot, Few-shot, Chain-of-Thought
- LangChain prompt templates
- Output parsers (String, JSON)
- RAG-specific prompt design
- Prompt management with external files

**Source Material:** Class 37 (Prompting)

---

## 1. Prompting Techniques Overview

| Technique | Description | When to Use |
|-----------|-------------|-------------|
| **Zero-shot** | No examples, just instruction | Simple tasks, well-defined outputs |
| **Few-shot** | Provide examples to guide output | Complex formats, specific styles |
| **Chain-of-Thought (CoT)** | "Think step by step" | Reasoning, math, logic tasks |
| **Role-based** | Assign a persona to the LLM | Expert-level answers |
| **Structured Output** | Force specific format (JSON, etc.) | API responses, data extraction |

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant")
print("✅ Setup complete")

## 2. Zero-Shot Prompting

In [ ]:
# Zero-shot: No examples, just clear instructions
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a GenAI expert. Give concise, exam-ready answers."),
    ("human", "{question}")
])

chain = zero_shot_prompt | llm | StrOutputParser()

try:
    answer = chain.invoke({"question": "Define RAG in one sentence."})
    print(f"🤖 Zero-shot answer: {answer}")
except Exception as e:
    print(f"⚠️ Error: {e}")

## 3. Few-Shot Prompting

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Define examples
examples = [
    {"input": "What is FAISS?", "output": "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search of dense vectors, developed by Meta AI."},
    {"input": "What is ChromaDB?", "output": "ChromaDB is an open-source embedding database designed for AI applications, offering easy local setup with auto-persistence."},
]

# Example prompt template
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# Few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Final prompt with few-shot examples
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a GenAI expert. Answer in the same concise style as the examples."),
    few_shot_prompt,
    ("human", "{input}"),
])

chain = final_prompt | llm | StrOutputParser()

try:
    answer = chain.invoke({"input": "What is Pinecone?"})
    print(f"🤖 Few-shot answer: {answer}")
except Exception as e:
    print(f"⚠️ Error: {e}")

## 4. Chain-of-Thought (CoT) Prompting

In [ ]:
# Chain-of-Thought: Guide the model to reason step by step
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a logical reasoning expert. Think step by step before giving your final answer."),
    ("human", """Question: {question}

Let's think through this step by step:
1. First, identify the key concepts
2. Then, analyze the relationships
3. Finally, state your conclusion

Step-by-step reasoning:""")
])

chain = cot_prompt | llm | StrOutputParser()

try:
    answer = chain.invoke({"question": "Should a company with frequently changing internal documents use RAG or fine-tuning, and why?"})
    print(f"🤖 CoT answer:\n{answer}")
except Exception as e:
    print(f"⚠️ Error: {e}")

## 5. RAG-Specific Prompt Design

### The Perfect RAG Prompt Has:
1. **System role** — Define the assistant's behavior
2. **Context placeholder** — Where retrieved docs go
3. **Grounding instruction** — "Only use the provided context"
4. **Fallback instruction** — What to say if context doesn't help
5. **Question placeholder** — The user's query

In [ ]:
# The ideal RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based on the provided context.

RULES:
1. ONLY use information from the context below to answer
2. If the context doesn't contain the answer, say "Based on the provided documents, I don't have enough information to answer this question."
3. Do NOT make up information
4. Cite which part of the context supports your answer
5. Be concise but thorough"""),
    ("human", """Context:
{context}

Question: {question}

Answer:""")
])

print("✅ RAG prompt template created")
print("\n📋 Template variables:", rag_prompt.input_variables)

In [ ]:
# Test RAG prompt with mock context
chain = rag_prompt | llm | StrOutputParser()

context = """RAG (Retrieval-Augmented Generation) is a technique that enhances LLM responses by retrieving 
relevant documents from a knowledge base before generating an answer. The pipeline has two phases: 
indexing (offline) and querying (online). During indexing, documents are loaded, chunked, embedded, 
and stored in a vector database."""

try:
    answer = chain.invoke({
        "context": context,
        "question": "What are the two phases of RAG?"
    })
    print(f"🤖 {answer}")
except Exception as e:
    print(f"⚠️ Error: {e}")

## 6. Output Parsers

In [ ]:
# String Output Parser (most common)
from langchain_core.output_parsers import StrOutputParser

str_chain = rag_prompt | llm | StrOutputParser()
# Returns: plain string
print("StrOutputParser: Converts LLM output to a plain string")

In [ ]:
# JSON Output Parser
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction assistant. 
Always respond in valid JSON format with these fields:
{{"answer": "your answer", "confidence": "high/medium/low", "key_concepts": ["list", "of", "concepts"]}}"""),
    ("human", "{question}")
])

json_chain = json_prompt | llm | JsonOutputParser()

try:
    result = json_chain.invoke({"question": "What is RAG in GenAI?"})
    print(f"📊 JSON output:")
    print(f"   Answer: {result.get('answer', 'N/A')}")
    print(f"   Confidence: {result.get('confidence', 'N/A')}")
    print(f"   Key concepts: {result.get('key_concepts', [])}")
except Exception as e:
    print(f"⚠️ Error: {e}")

## 7. Prompt Management with External Files

In [ ]:
import json

# Create a prompts file
prompts = {
    "rag_qa": {
        "system": "Answer questions based only on the provided context. Be concise.",
        "template": "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    },
    "summarize": {
        "system": "You are a summarization expert.",
        "template": "Summarize the following text in {num_sentences} sentences:\n\n{text}"
    },
    "extract": {
        "system": "Extract structured information from text. Respond in JSON.",
        "template": "Text: {text}\n\nExtract the following fields: {fields}"
    }
}

# Save to file
with open("./data/prompts.json", 'w') as f:
    json.dump(prompts, f, indent=2)

print("✅ Prompts saved to data/prompts.json")
print(f"📋 Available prompts: {list(prompts.keys())}")

In [ ]:
# Load and use prompts from file
with open("./data/prompts.json", 'r') as f:
    loaded_prompts = json.load(f)

# Use the RAG QA prompt
qa_config = loaded_prompts["rag_qa"]
prompt = ChatPromptTemplate.from_messages([
    ("system", qa_config["system"]),
    ("human", qa_config["template"])
])

print(f"✅ Loaded '{qa_config['system'][:50]}...'")
print(f"📋 Variables: {prompt.input_variables}")

### 🎯 Exam Tip
> **Why manage prompts externally?**
> - **Reusability** — Same prompt across multiple chains
> - **Version control** — Track prompt changes over time
> - **A/B testing** — Easily swap prompts without code changes
> - **Separation of concerns** — Prompts are content, code is logic

## 8. Prompt Best Practices for RAG

| Practice | Why |
|----------|-----|
| **Be specific** | Vague prompts → vague answers |
| **Use delimiters** | Separate context, question, and instructions clearly |
| **Add grounding rules** | "Only use provided context" prevents hallucination |
| **Include fallback** | "If you don't know, say so" prevents made-up answers |
| **Specify format** | Tell the LLM exactly how to structure the output |
| **Use negative instructions** | "Do NOT make up information" is powerful |
| **Set temperature** | 0 for factual, higher for creative |
| **Test with edge cases** | What if context is empty? Irrelevant? Contradictory? |

### Temperature Guide
```
temperature=0.0 → Deterministic, factual (RAG, data extraction)
temperature=0.3 → Slightly creative (summaries, paraphrasing)
temperature=0.7 → Creative (writing, brainstorming)
temperature=1.0 → Maximum creativity (poetry, stories)
```

## 🧠 Self-Assessment Quiz

---

**Q1.** What is the difference between zero-shot and few-shot prompting?

<details>
<summary>Click for Answer</summary>

- **Zero-shot**: No examples provided — relies entirely on the model's training  
- **Few-shot**: Provides examples of input-output pairs to guide the model's format and style  
Few-shot is better when you need a specific output format or style.
</details>

---

**Q2.** In a RAG prompt, why is the instruction "If the context doesn't contain the answer, say so" important?

<details>
<summary>Click for Answer</summary>

Without this fallback, the LLM will **hallucinate** — generate plausible-sounding but incorrect information. The grounding instruction forces the LLM to admit when it doesn't have enough context, which is crucial for trustworthy RAG systems.
</details>

---

**Q3.** What temperature setting should you use for a factual RAG QA system?

<details>
<summary>Click for Answer</summary>

**temperature=0** (or close to 0). Low temperature makes the model more deterministic and factual, which is essential for RAG where accuracy matters more than creativity.
</details>

---

**Q4.** What does `ChatPromptTemplate.from_messages()` accept?

<details>
<summary>Click for Answer</summary>

A list of tuples in the format `(role, content)` where:  
- `"system"` — system instructions  
- `"human"` — user message  
- `"ai"` — assistant response (for few-shot examples)  

Example: `[("system", "You are..."), ("human", "{question}")]`
</details>

---

**Q5.** What is the difference between `StrOutputParser` and `JsonOutputParser`?

<details>
<summary>Click for Answer</summary>

- `StrOutputParser()` — Returns the LLM output as a **plain string**  
- `JsonOutputParser()` — Parses the LLM output as **JSON** and returns a Python dict  
Use JSON parser when you need structured data; use string parser for free-form text.
</details>

---

## ✅ Module 8 Complete!

**Key Takeaways:**
1. Zero-shot = no examples, Few-shot = with examples, CoT = step-by-step reasoning
2. RAG prompts need: grounding rules, fallback, context placeholder, clear format
3. `StrOutputParser` for text, `JsonOutputParser` for structured data
4. Store prompts in external JSON files for reusability and version control
5. Use `temperature=0` for factual RAG systems
6. Negative instructions ("Do NOT...") are powerful for preventing hallucination

**Next:** [Module 09 — Evaluation & Optimization](./09_Evaluation_and_Optimization.md)